# Part 2 — 8-Week Trailing Baseline & Incremental Lift

**Goal:** Establish what "normal" looked like *before* the promo, then isolate how much the SPRING20 event moved the needle.

---

## How This Notebook Is Structured

1. **Load** the cleaned data from Part 1
2. **Build the baseline** (Feb 28 – Apr 24, 2025) — daily averages for orders, revenue, AOV, UPT
3. **Build the promo picture** using **Option C** — two views side by side:
   - *Narrow view:* SPRING20-coded orders only (code-driven lift)
   - *Broad view:* All Apr 25–27 orders (total day performance)
4. **Lift table** — absolute and % lift for every metric
5. **Weekend-adjusted baseline** — because Apr 25–27 was Fri–Sun, not a typical mix of days
6. **Pull-forward check** — did orders dip in the days before the promo?
7. **Caveats** — what the numbers can and can't tell you

> **Why Option C?**  
> Not all Apr 25–27 orders carried the SPRING20 code — organic orders ran alongside promo orders.  
> Reporting only the code view understates total day performance.  
> Reporting only the date view conflates promo-driven and organic demand.  
> Showing both gives the full picture and lets stakeholders choose the right lens for their question.


## 1. Imports & Constants

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)

# ── Key dates ──────────────────────────────────────────────────────────────────
BASELINE_START = pd.Timestamp("2025-02-28")
BASELINE_END   = pd.Timestamp("2025-04-24")   # day before promo
PROMO_START    = pd.Timestamp("2025-04-25")   # Friday
PROMO_END      = pd.Timestamp("2025-04-27")   # Sunday
PROMO_CODE     = "SPRING20"

BASELINE_DAYS  = (BASELINE_END - BASELINE_START).days + 1   # should be 56
PROMO_DAYS     = (PROMO_END - PROMO_START).days + 1         # 3

print(f"Baseline window : {BASELINE_START.date()} → {BASELINE_END.date()} ({BASELINE_DAYS} days)")
print(f"Promo window    : {PROMO_START.date()} → {PROMO_END.date()} ({PROMO_DAYS} days)")


Baseline window : 2025-02-28 → 2025-04-24 (56 days)
Promo window    : 2025-04-25 → 2025-04-27 (3 days)


## 2. Load Cleaned Data

Using `orders_clean.csv` produced by Part 1. If you haven't run Part 1 yet, do that first.


In [2]:
df = pd.read_csv("/Users/apple/Desktop/Assignment2/Output/orders_clean.csv", parse_dates=["order_date"])

print(f"Loaded: {len(df):,} orders  |  {df['order_date'].min().date()} → {df['order_date'].max().date()}")
df.head(3)


Loaded: 5,678 orders  |  2025-02-14 → 2025-04-30


,order_id,order_date,customer_type,discount_code,net_sales,units,aov,upt,skus
0,ord_100000,2025-02-14,returning,WELCOME10,55.80,1,55.80,1,CYM-MAG-30
1,ord_100001,2025-02-14,subscriber,NaN,139.65,2,139.65,2,CYM-PRO-30|CYM-COL-30
2,ord_100002,2025-02-14,new,WELCOME10,110.70,2,110.70,2,CYM-PRO-30|CYM-MTB-30


## 3. Build the 8-Week Baseline (Feb 28 – Apr 24)

The baseline answers: **what did a normal day look like before the promo?**

### What counts as baseline?

We include **all orders** in this window — including those with other discount codes  
(WELCOME10, FRIEND10, SUB15, etc.).

**Why?** Because that mixed picture *is* what normal looked like. Your business always  
has some coupon usage. Stripping those out would artificially depress the baseline  
and inflate the lift. We want "normal days" not "perfect days."


In [3]:
baseline_df = df[
    (df["order_date"] >= BASELINE_START) &
    (df["order_date"] <= BASELINE_END)
].copy()

print(f"Baseline orders: {len(baseline_df):,} across {BASELINE_DAYS} days")
print()

# Aggregate to daily level first — then average across days
# This is the correct approach: avg(daily totals), NOT sum/N_days of all orders
daily_baseline = (
    baseline_df
    .groupby("order_date")
    .agg(
        orders    = ("order_id",   "count"),
        net_sales = ("net_sales",  "sum"),
        units     = ("units",      "sum"),
    )
    .reset_index()
)

# AOV and UPT computed at the daily aggregate level
daily_baseline["aov"] = daily_baseline["net_sales"] / daily_baseline["orders"]
daily_baseline["upt"] = daily_baseline["units"]     / daily_baseline["orders"]

daily_baseline.head(5)


Baseline orders: 3,962 across 56 days



,order_date,orders,net_sales,units,aov,upt
0,2025-02-28,97,"12,476.65",217,128.63,2.24
1,2025-03-01,70,"8,361.73",145,119.45,2.07
2,2025-03-02,61,"7,233.61",126,118.58,2.07
3,2025-03-03,71,"7,525.62",131,105.99,1.85
4,2025-03-04,57,"6,336.66",107,111.17,1.88


In [4]:
# Sanity check: are all 56 days present?
actual_days = len(daily_baseline)
print(f"Expected baseline days : {BASELINE_DAYS}")
print(f"Days with orders found : {actual_days}")

if actual_days < BASELINE_DAYS:
    missing = pd.date_range(BASELINE_START, BASELINE_END).difference(daily_baseline["order_date"])
    print(f"\n⚠️  {len(missing)} days with zero orders:")
    print([str(d.date()) for d in missing])
    # Fill missing days with zeros so the average reflects true "daily" performance
    zero_days = pd.DataFrame({"order_date": missing, "orders": 0, "net_sales": 0.0, "units": 0, "aov": np.nan, "upt": np.nan})
    daily_baseline = pd.concat([daily_baseline, zero_days]).sort_values("order_date").reset_index(drop=True)
    print(f"Zero-filled. Total rows: {len(daily_baseline)}")


Expected baseline days : 56
Days with orders found : 56


In [5]:
# Baseline summary stats
baseline_stats = {
    "Avg daily orders"      : daily_baseline["orders"].mean(),
    "Avg daily net sales ($)": daily_baseline["net_sales"].mean(),
    "Baseline AOV ($)"      : baseline_df["net_sales"].sum() / len(baseline_df),
    "Baseline UPT"          : baseline_df["units"].sum()     / len(baseline_df),
    "Std dev daily orders"  : daily_baseline["orders"].std(),
    "Std dev daily revenue" : daily_baseline["net_sales"].std(),
}

print("=== 8-WEEK BASELINE (Feb 28 – Apr 24) ===")
for k, v in baseline_stats.items():
    print(f"  {k:<35} {v:>10.2f}")


=== 8-WEEK BASELINE (Feb 28 – Apr 24) ===
  Avg daily orders                         70.75
  Avg daily net sales ($)                8370.55
  Baseline AOV ($)                        118.31
  Baseline UPT                              2.05
  Std dev daily orders                     12.17
  Std dev daily revenue                  1585.44


## 4. Weekend-Adjusted Baseline

**This is important.** Apr 25–27 was a **Friday, Saturday, Sunday**.  
A simple daily average across all 56 baseline days mixes weekday and weekend behaviour.  
Some of the "lift" we'd calculate is just the natural weekend bump — not the promo.

To isolate the true promo effect, we also compute a baseline using **only Fri/Sat/Sun  
days** from the baseline window. This gives an apples-to-apples comparison.


In [6]:
daily_baseline["day_of_week"] = daily_baseline["order_date"].dt.day_name()
daily_baseline["is_weekend_like"] = daily_baseline["day_of_week"].isin(["Friday", "Saturday", "Sunday"])

weekend_baseline = daily_baseline[daily_baseline["is_weekend_like"]]

print(f"Fri/Sat/Sun days in baseline window: {len(weekend_baseline)}")
print()

# How many of each day type?
print(daily_baseline["day_of_week"].value_counts().sort_index())


Fri/Sat/Sun days in baseline window: 24

day_of_week
Friday       8
Monday       8
Saturday     8
Sunday       8
Thursday     8
Tuesday      8
Wednesday    8
Name: count, dtype: int64


In [7]:
# Day-of-week average orders — shows natural weekly rhythm
dow_avg = (
    daily_baseline
    .groupby("day_of_week")["orders"]
    .mean()
    .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
)

print("=== AVG DAILY ORDERS BY DAY OF WEEK (baseline) ===")
for day, avg in dow_avg.items():
    bar = "█" * int(avg / 2)
    print(f"  {day:<12} {avg:>6.1f}  {bar}")


=== AVG DAILY ORDERS BY DAY OF WEEK (baseline) ===
  Monday         71.0  ███████████████████████████████████
  Tuesday        72.9  ████████████████████████████████████
  Wednesday      67.4  █████████████████████████████████
  Thursday       76.5  ██████████████████████████████████████
  Friday         84.9  ██████████████████████████████████████████
  Saturday       65.4  ████████████████████████████████
  Sunday         57.2  ████████████████████████████


In [8]:
weekend_stats = {
    "Avg daily orders (Fri/Sat/Sun)"      : weekend_baseline["orders"].mean(),
    "Avg daily net sales ($) (Fri/Sat/Sun)": weekend_baseline["net_sales"].mean(),
    "AOV (Fri/Sat/Sun) ($)"               : (weekend_baseline["net_sales"].sum() /
                                              weekend_baseline["orders"].sum()),
    "UPT (Fri/Sat/Sun)"                   : (weekend_baseline["units"].sum() /
                                              weekend_baseline["orders"].sum()
                                              if "units" in weekend_baseline.columns else np.nan),
}

print("=== WEEKEND-ADJUSTED BASELINE (Fri/Sat/Sun only) ===")
for k, v in weekend_stats.items():
    print(f"  {k:<45} {v:>10.2f}")


=== WEEKEND-ADJUSTED BASELINE (Fri/Sat/Sun only) ===
  Avg daily orders (Fri/Sat/Sun)                     69.17
  Avg daily net sales ($) (Fri/Sat/Sun)            8118.17
  AOV (Fri/Sat/Sun) ($)                             117.37
  UPT (Fri/Sat/Sun)                                   2.03


## 5. Build the Promo Picture — Option C

We report two views:

| View | Definition | Use for |
|---|---|---|
| **Narrow** | SPRING20 code only | Measuring code-driven, attributable lift |
| **Broad** | All Apr 25–27 orders | Measuring total day performance |

The **gap between them** = organic orders on promo days (customers who bought without the code).


In [9]:
apr_window = df[
    (df["order_date"] >= PROMO_START) &
    (df["order_date"] <= PROMO_END)
].copy()

# Narrow: SPRING20 coded orders only
promo_narrow = apr_window[apr_window["discount_code"] == PROMO_CODE]

# Broad: all orders in the window
promo_broad  = apr_window

print(f"Apr 25–27 total orders      : {len(promo_broad):,}")
print(f"  → With SPRING20 code      : {len(promo_narrow):,}  ({len(promo_narrow)/len(promo_broad)*100:.1f}%)")
print(f"  → Without SPRING20 code   : {len(promo_broad)-len(promo_narrow):,}  ({(len(promo_broad)-len(promo_narrow))/len(promo_broad)*100:.1f}%)")


Apr 25–27 total orders      : 467
  → With SPRING20 code      : 357  (76.4%)
  → Without SPRING20 code   : 110  (23.6%)


In [10]:
def daily_avg_metrics(orders_df, n_days):
    """Compute daily-average metrics from a set of orders over n_days."""
    return {
        "avg_daily_orders"    : len(orders_df) / n_days,
        "avg_daily_net_sales" : orders_df["net_sales"].sum() / n_days,
        "aov"                 : orders_df["net_sales"].sum() / max(len(orders_df), 1),
        "upt"                 : orders_df["units"].sum()     / max(len(orders_df), 1),
    }

metrics_baseline         = daily_avg_metrics(baseline_df,   BASELINE_DAYS)
metrics_baseline_weekend = {
    "avg_daily_orders"    : weekend_baseline["orders"].mean(),
    "avg_daily_net_sales" : weekend_baseline["net_sales"].mean(),
    "aov"                 : weekend_baseline["net_sales"].sum() / weekend_baseline["orders"].sum(),
    "upt"                 : weekend_baseline["units"].sum()     / weekend_baseline["orders"].sum()
                            if "units" in weekend_baseline.columns else np.nan,
}
metrics_narrow  = daily_avg_metrics(promo_narrow, PROMO_DAYS)
metrics_broad   = daily_avg_metrics(promo_broad,  PROMO_DAYS)

print("Metrics computed for all four scenarios.")


Metrics computed for all four scenarios.


## 6. Lift Table

For each metric we compute:
- **Absolute lift** = promo daily avg − baseline daily avg
- **% lift** = absolute lift / baseline daily avg × 100

We show four comparisons so stakeholders can pick the right lens.


In [11]:
def lift_row(label, base, promo_val):
    abs_lift = promo_val - base
    pct_lift = (abs_lift / base * 100) if base != 0 else np.nan
    return {"Metric": label, "Baseline": base, "Promo": promo_val,
            "Abs Lift": abs_lift, "% Lift": pct_lift}

def build_lift_table(base_metrics, promo_metrics, base_label, promo_label):
    rows = [
        lift_row("Avg daily orders",       base_metrics["avg_daily_orders"],    promo_metrics["avg_daily_orders"]),
        lift_row("Avg daily net sales ($)", base_metrics["avg_daily_net_sales"], promo_metrics["avg_daily_net_sales"]),
        lift_row("AOV ($)",                base_metrics["aov"],                 promo_metrics["aov"]),
        lift_row("UPT",                    base_metrics["upt"],                 promo_metrics["upt"]),
    ]
    t = pd.DataFrame(rows).set_index("Metric")
    t.columns.name = f"{base_label} → {promo_label}"
    return t.round(2)

print("=" * 65)
print("COMPARISON 1: All-day baseline vs SPRING20 orders (attributable lift)")
print("=" * 65)
t1 = build_lift_table(metrics_baseline, metrics_narrow, "All-day baseline", "SPRING20 only")
print(t1.to_string())

print()
print("=" * 65)
print("COMPARISON 2: All-day baseline vs All Apr 25-27 orders (total window)")
print("=" * 65)
t2 = build_lift_table(metrics_baseline, metrics_broad, "All-day baseline", "All promo-window")
print(t2.to_string())

print()
print("=" * 65)
print("COMPARISON 3: Weekend baseline vs SPRING20 orders (day-of-week adjusted)")
print("=" * 65)
t3 = build_lift_table(metrics_baseline_weekend, metrics_narrow, "Weekend baseline", "SPRING20 only")
print(t3.to_string())

print()
print("=" * 65)
print("COMPARISON 4: Weekend baseline vs All Apr 25-27 orders")
print("=" * 65)
t4 = build_lift_table(metrics_baseline_weekend, metrics_broad, "Weekend baseline", "All promo-window")
print(t4.to_string())


COMPARISON 1: All-day baseline vs SPRING20 orders (attributable lift)
All-day baseline → SPRING20 only  Baseline     Promo  Abs Lift  % Lift
Metric                                                                
Avg daily orders                     70.75    119.00     48.25   68.20
Avg daily net sales ($)           8,370.55 11,875.83  3,505.27   41.88
AOV ($)                             118.31     99.80    -18.51  -15.65
UPT                                   2.05      2.11      0.06    3.09

COMPARISON 2: All-day baseline vs All Apr 25-27 orders (total window)
All-day baseline → All promo-window  Baseline     Promo  Abs Lift  % Lift
Metric                                                                   
Avg daily orders                        70.75    155.67     84.92  120.02
Avg daily net sales ($)              8,370.55 16,139.91  7,769.35   92.82
AOV ($)                                118.31    103.68    -14.63  -12.36
UPT                                      2.05      2.09      0.

In [12]:
# Save all four tables to CSV
with pd.ExcelWriter("/Users/apple/Desktop/Assignment2/Output/part2_lift_tables.xlsx") as writer:
    t1.to_excel(writer, sheet_name="1_allbase_vs_spring20")
    t2.to_excel(writer, sheet_name="2_allbase_vs_allwindow")
    t3.to_excel(writer, sheet_name="3_wkndbase_vs_spring20")
    t4.to_excel(writer, sheet_name="4_wkndbase_vs_allwindow")

print("Saved → part2_lift_tables.xlsx")


Saved → part2_lift_tables.xlsx


## 7. Pull-Forward Check

**What is pull-forward?** Customers who know a sale is coming may delay buying  
until the promo starts. This means:
- Orders dip *just before* the promo
- Promo orders are inflated by demand that was "borrowed" from pre-promo days
- The true incremental lift is lower than the raw numbers suggest

We check the 7 days before the promo (Apr 18–24) vs the broader baseline average.


In [13]:
PRE_PROMO_START = pd.Timestamp("2025-04-18")
PRE_PROMO_END   = pd.Timestamp("2025-04-24")

pre_promo_df = df[
    (df["order_date"] >= PRE_PROMO_START) &
    (df["order_date"] <= PRE_PROMO_END)
]

daily_pre = (
    pre_promo_df
    .groupby("order_date")
    .agg(orders=("order_id","count"), net_sales=("net_sales","sum"))
    .reset_index()
)

baseline_avg_orders = metrics_baseline["avg_daily_orders"]
baseline_avg_sales  = metrics_baseline["avg_daily_net_sales"]

print("=== PRE-PROMO DAILY ORDERS (Apr 18–24) vs Baseline Avg ===")
print(f"{'Date':<14} {'Orders':>8} {'vs Baseline':>12} {'Net Sales ($)':>14} {'vs Baseline':>12}")
print("-" * 65)
for _, row in daily_pre.iterrows():
    o_diff = row["orders"]    - baseline_avg_orders
    s_diff = row["net_sales"] - baseline_avg_sales
    flag   = " ⬇" if o_diff < -5 else ""
    print(f"{str(row['order_date'].date()):<14} {row['orders']:>8.0f}"
          f" {o_diff:>+11.1f}  {row['net_sales']:>13,.0f} {s_diff:>+11.0f}{flag}")

print("-" * 65)
print(f"{'Baseline avg':<14} {baseline_avg_orders:>8.1f}  {'—':>12}  {baseline_avg_sales:>13,.0f}  {'—':>11}")


=== PRE-PROMO DAILY ORDERS (Apr 18–24) vs Baseline Avg ===
Date             Orders  vs Baseline  Net Sales ($)  vs Baseline
-----------------------------------------------------------------
2025-04-18           63        -7.8          6,845       -1525 ⬇
2025-04-19           71        +0.2          8,594        +223
2025-04-20           49       -21.8          5,600       -2770 ⬇
2025-04-21           76        +5.2          8,471        +101
2025-04-22           78        +7.2          8,847        +476
2025-04-23           75        +4.2          9,868       +1497
2025-04-24           89       +18.2         10,406       +2036
-----------------------------------------------------------------
Baseline avg       70.8             —          8,371            —


In [14]:
avg_prepromo_orders = daily_pre["orders"].mean()
pullforward_signal  = baseline_avg_orders - avg_prepromo_orders

print(f"Baseline avg daily orders    : {baseline_avg_orders:.1f}")
print(f"Pre-promo avg daily orders   : {avg_prepromo_orders:.1f}")
print(f"Difference                   : {pullforward_signal:+.1f} orders/day")
print()

if pullforward_signal > 5:
    print("⚠️  Possible pull-forward effect: orders dropped noticeably before the promo.")
    print("   Some promo volume may be borrowed demand, not truly incremental.")
elif pullforward_signal < -5:
    print("ℹ️  Orders were actually ABOVE baseline pre-promo — no pull-forward evident.")
    print("   Pre-promo period may have had its own organic momentum.")
else:
    print("✓  No significant pull-forward detected. Pre-promo orders tracked close to baseline.")


Baseline avg daily orders    : 70.8
Pre-promo avg daily orders   : 71.6
Difference                   : -0.8 orders/day

✓  No significant pull-forward detected. Pre-promo orders tracked close to baseline.


## 8. Caveats — What the Numbers Can and Can't Tell You

These are not disclaimers — they are the things a stakeholder *must* understand  
before acting on the lift figures above.

---

### Caveat 1: Day-of-week mix
The baseline average is computed across all 7 day types (56 days, ~8 of each).  
Apr 25–27 was **Fri–Sat–Sun** — days that typically see higher order volume anyway.  
**Use Comparisons 3 & 4 (weekend-adjusted)** for the most defensible lift estimate.  
The difference between Comparison 1 and Comparison 3 tells you how much of the  
apparent lift is just "weekend effect."

---

### Caveat 2: Attribution — code vs window
Not all Apr 25–27 orders used SPRING20. Organic customers bought on those days too.  
- **If the question is "did the promo code work?"** → use Comparison 1 (SPRING20 only)  
- **If the question is "how did those days perform?"** → use Comparison 2 (all orders)  
- Conflating the two overstates or understates depending on which way you conflate.

---

### Caveat 3: Pull-forward demand
If orders dipped before the promo, some of the lift is *borrowed* from adjacent days,  
not truly incremental. Check the pull-forward cell above. If the dip is material,  
net incremental lift = promo lift − (baseline avg − pre-promo avg) × days affected.

---

### Caveat 4: Cannibalization of margin
This analysis measures **order volume and revenue lift**, not **profit lift**.  
A 20% discount means each SPRING20 order generated ~20% less margin than a baseline  
order of the same size. High order lift + high discount depth can still be margin-negative.  
Parts 5–6 address this at the SKU level.

---

### Caveat 5: Sample size
The promo window is only **3 days**. Day-to-day variance is high — a single large B2B  
order or a site outage on one of those days can move the averages meaningfully.  
The baseline (56 days) is much more stable. Treat promo-day averages as directional,  
not statistically precise.


## 9. Summary

| | All-day baseline | Weekend baseline |
|---|---|---|
| **SPRING20 orders lift (orders)** | see t1 | see t3 |
| **All Apr 25–27 lift (orders)** | see t2 | see t4 |

**The number to lead with in a presentation:**  
*Weekend-adjusted baseline vs SPRING20 orders* (Comparison 3) — it controls for day-of-week  
and attributes lift only to code-driven orders. It is the most conservative and most defensible.

**The number Finance will ask about:**  
*All-day baseline vs all Apr 25–27 orders* (Comparison 2) — total revenue those days vs normal.

**Next:** Part 3 — Simple projection for a future promo using these baseline and lift figures.
